In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
import os
import glob
import shap
import json
import networkx as nx
import seaborn as sns
import lingam
import warnings

from sklearn.preprocessing import (
    LabelEncoder,
    StandardScaler,
    MinMaxScaler,
)

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from typing import Dict
from sklearn.model_selection import cross_val_score, KFold
from sklearn.model_selection import train_test_split
from sklearn.model_selection import TimeSeriesSplit
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, f1_score
from scipy.stats import pearsonr
from statsmodels.tsa.stattools import ccf
from statsmodels.stats.stattools import durbin_watson
from statsmodels.tsa.api import VAR
from scipy import stats


from sklearn.metrics import (
    roc_auc_score, 
    roc_curve, 
    confusion_matrix, 
    precision_score, 
    recall_score, 
    accuracy_score,
    classification_report
)


warnings.filterwarnings('ignore')
print("✅ Imports loaded successfully!")



✅ Imports loaded successfully!


<br> <br> <br>

## Create Causal Graph 

Create a causal graph on machine_98 dataset

In [2]:
def create_causal_network(df, features, regularize=True, noise_level=1e-4):
    X_selected = df[features].values

    # Initialize the VARLiNGAM model
    model = lingam.VARLiNGAM()

    # Attempt to fit the model; add noise if a LinAlgError occurs
    try:
        model.fit(X_selected)
    except np.linalg.LinAlgError as e:
        print("LinAlgError encountered during model fitting:", e)
        if regularize:
            print(f"Applying regularization: adding noise (std={noise_level}) to the data and trying again.")
            X_selected += np.random.normal(0, noise_level, X_selected.shape)
            model.fit(X_selected)
        else:
            raise e

    # Retrieve the adjacency matrices (one per lag)
    adjacency_matrices = model.adjacency_matrices_

    # Build a directed graph and create a list of dictionaries for each non-zero edge.
    G = nx.DiGraph()
    G.add_nodes_from(features)
    n_features = len(features)
    structured_adjacency = []

    # Loop through each adjacency matrix (one per lag)
    for lag_index, adj_matrix in enumerate(adjacency_matrices):
        for i in range(n_features):
            for j in range(n_features):
                weight = adj_matrix[i, j]
                if weight != 0:
                    edge_dict = {
                        "source_feature": features[i],
                        "target_feature": features[j],
                        "effect_strength": weight,
                    }
                    structured_adjacency.append(edge_dict)
                    G.add_edge(features[i], features[j], weight=weight)

    # Plot the graph using a spring layout for clarity
    pos = nx.spring_layout(G, seed=42)  # seed for reproducibility
    fig, ax = plt.subplots(figsize=(12, 8))
    nx.draw(
        G,
        pos,
        with_labels=True,
        node_color="lightblue",
        node_size=1500,
        arrowstyle="->",
        arrowsize=20,
        edge_color="gray",
        font_size=10,
        ax=ax,
    )

    # Display the edge weights formatted to two decimal places
    edge_labels = nx.get_edge_attributes(G, "weight")
    edge_labels = {edge: f"{weight:.2f}" for edge, weight in edge_labels.items()}
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_color="red", ax=ax)

    ax.set_title("Causal Network Graph from VARLiNGAM")
    ax.axis("off")
    # plt.show()
    plt.close()

    return structured_adjacency, fig


In [ ]:
df_machine_98 = pd.read_csv("../../data/azure_pm/lag_features/machine_98_lag_features.csv")
df_machine_98.drop(columns=["datetime", "failure", "comp"], inplace=True)



## Create causal graph
structured_adjacency, fig = create_causal_network(df=df_machine_98, features=df_machine_98.columns)

In [6]:
# Ensure the target directory exists
output_dir = "../../causality_graph/azure_pm/machine_98/"
os.makedirs(output_dir, exist_ok=True)

# Save structured_adjacency as JSON
with open(os.path.join(output_dir, "structured_adjacency.json"), "w") as f:
    json.dump(structured_adjacency, f, indent=2)

# Save the figure as PNG
fig.savefig(os.path.join(output_dir, "causal_network.png"))

---

<br> <br>

### Read Causal graph file

In [5]:
with open("../../causality_graph/azure_pm/machine_98/structured_adjacency.json", "r") as f:
    structured_adjacency = json.load(f)

print(f"Loaded {len(structured_adjacency)} edges from structured_adjacency.json")

Loaded 774 edges from structured_adjacency.json


<br> <br> 


## Apply machine learning

Apply machine leanirng on machine_98 dataset

In [7]:
def walk_forward_validation(df, n_splits=5):
    """
    Perform walk-forward validation with expanding window for MULTICLASS/CATEGORICAL classification.
    
    Args:
        df: DataFrame with features and CATEGORICAL target
        n_splits: Number of validation splits
        
    Returns:
        Dictionary with results for each model
    """
    print(f"🚀 Starting Walk-Forward Validation with {n_splits} splits...")
    print("=" * 60)
    
    # Prepare features and target
    exclude_cols = ['datetime', 'machineID', 'failure', 'binary_failure', 'target']
    feature_cols = [col for col in df.columns if col not in exclude_cols]
    
    X = df[feature_cols].fillna(0)  # Fill any remaining NaN
    y_original = df['target']
    
    # FIXED: Remap target classes to be consecutive for XGBoost compatibility
    unique_classes = sorted(y_original.unique())
    n_classes = len(unique_classes)
    
    # Create mapping from original classes to consecutive classes
    class_mapping = {original_class: consecutive_class for consecutive_class, original_class in enumerate(unique_classes)}
    reverse_mapping = {consecutive_class: original_class for original_class, consecutive_class in class_mapping.items()}
    
    # Apply mapping to target
    y = y_original.map(class_mapping)
    
    print(f"📊 Dataset info:")
    print(f"   - Total samples: {len(X):,}")
    print(f"   - Features: {len(feature_cols)}")
    print(f"   - Original target classes: {unique_classes}")
    print(f"   - Remapped target classes: {sorted(y.unique())}")
    print(f"   - Class mapping: {class_mapping}")
    print(f"   - Number of classes: {n_classes}")
    print(f"   - Target distribution: {y.value_counts().to_dict()}")
    
    if n_classes < 2:
        print(f"❌ ERROR: Need at least 2 classes for classification, but found: {n_classes}")
        return None, None
    
    # Determine if binary or multiclass
    is_binary = n_classes == 2 and set(unique_classes) == {0, 1}
    classification_type = "BINARY" if is_binary else "MULTICLASS"
    
    print(f"🎯 Classification type: {classification_type}")
    
    # Model configurations
    models = {
        'RandomForest': RandomForestClassifier(
            n_estimators=100, max_depth=10, min_samples_split=5,
            random_state=42, n_jobs=-1
        ),
        'LightGBM': LGBMClassifier(
            n_estimators=100, max_depth=6, learning_rate=0.1,
            random_state=42, verbose=-1, force_col_wise=True
        ),
        'XGBoost': XGBClassifier(
            n_estimators=100, max_depth=6, learning_rate=0.1,
            random_state=42, eval_metric='logloss', verbosity=0
        ),
        'CatBoost': CatBoostClassifier(
            iterations=100, depth=6, learning_rate=0.1,
            random_state=42, verbose=False, allow_writing_files=False
        )
    }
    
    # Enhanced results storage - Modified for multiclass
    results = {name: {
        'auc': [],  # Will be macro-averaged AUC for multiclass
        'precision': [], 'recall': [], 'accuracy': [], 
        'f1_score': [],  # NEW: F1 score (important for multiclass)
        'macro_precision': [],  # NEW: Macro-averaged precision
        'macro_recall': [],  # NEW: Macro-averaged recall
        'weighted_precision': [],  # NEW: Weighted precision
        'weighted_recall': [],  # NEW: Weighted recall
        'per_class_metrics': [],  # NEW: Per-class precision, recall, f1
        'confusion_matrices': [],  # NEW: Confusion matrices
        'predictions': [], 'y_true': [], 'pred_probabilities': []  # Modified
    } for name in models.keys()}
    
    # For binary classification, keep the enhanced ROC metrics
    if is_binary:
        for name in results.keys():
            results[name].update({
                'normalized_auc': [],
                'optimal_threshold': [],
                'optimal_precision': [],
                'optimal_recall': [],
                'youden_index': [],
                'roc_curves': []
            })
    
    # Create time series splits (expanding window)
    tscv = TimeSeriesSplit(n_splits=n_splits)
    
    fold = 1
    for train_idx, test_idx in tscv.split(X):
        print(f"\n📅 Fold {fold}/{n_splits}")
        print("-" * 30)
        
        # Split data
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        
        # Check classes in training and test sets
        train_classes = sorted(y_train.unique())
        test_classes = sorted(y_test.unique())
        
        if len(train_classes) < 2:
            print(f"⚠️  Skipping fold {fold}: Training set has only one class: {train_classes}")
            fold += 1
            continue
        
        print(f"   Train: {len(X_train):,} samples, classes: {train_classes}")
        print(f"   Test:  {len(X_test):,} samples, classes: {test_classes}")
        print(f"   Train distribution: {y_train.value_counts().to_dict()}")
        print(f"   Test distribution: {y_test.value_counts().to_dict()}")
        
        # Calculate class weights for imbalanced data
        try:
            class_weights = compute_class_weight(
                'balanced', classes=np.unique(y_train), y=y_train
            )
            class_weight_dict = {cls: weight for cls, weight in zip(np.unique(y_train), class_weights)}
            
            # For XGBoost multiclass, we'll use class weights differently
            if is_binary and len(class_weights) > 1:
                scale_pos_weight = class_weights[1] / class_weights[0]
            else:
                scale_pos_weight = 1
        except:
            class_weight_dict = None
            scale_pos_weight = 1
        
        # Train each model
        for model_name, base_model in models.items():
            try:
                # Configure model with class weights
                if model_name == 'RandomForest':
                    model = RandomForestClassifier(
                        n_estimators=100, max_depth=10, min_samples_split=5,
                        class_weight='balanced', random_state=42, n_jobs=-1
                    )
                elif model_name == 'LightGBM':
                    model = LGBMClassifier(
                        n_estimators=100, max_depth=6, learning_rate=0.1,
                        class_weight='balanced', random_state=42, 
                        verbose=-1, force_col_wise=True
                    )
                elif model_name == 'XGBoost':
                    if is_binary:
                        model = XGBClassifier(
                            n_estimators=100, max_depth=6, learning_rate=0.1,
                            scale_pos_weight=scale_pos_weight, random_state=42,
                            eval_metric='logloss', verbosity=0
                        )
                    else:
                        # For multiclass, XGBoost automatically handles it
                        # FIXED: Ensure classes are consecutive
                        model = XGBClassifier(
                            n_estimators=100, max_depth=6, learning_rate=0.1,
                            random_state=42, eval_metric='mlogloss', verbosity=0, 
                            num_classes=n_classes
                        )
                elif model_name == 'CatBoost':
                    if is_binary:
                        model = CatBoostClassifier(
                            iterations=100, depth=6, learning_rate=0.1,
                            class_weights=[1, scale_pos_weight], random_state=42,
                            verbose=False, allow_writing_files=False
                        )
                    else:
                        # For multiclass, use auto class weights
                        model = CatBoostClassifier(
                            iterations=100, depth=6, learning_rate=0.1,
                            auto_class_weights='Balanced', random_state=42,
                            verbose=False, allow_writing_files=False
                        )
                
                # Train model
                model.fit(X_train, y_train)
                
                # Make predictions
                y_pred = model.predict(X_test)
                y_pred_proba = model.predict_proba(X_test)
                
                # Calculate metrics based on classification type
                if is_binary:
                    # Binary classification metrics (existing logic)
                    y_pred_proba_pos = y_pred_proba[:, 1]  # Positive class probabilities
                    
                    try:
                        if len(y_test.unique()) > 1:
                            # Standard AUC
                            auc = roc_auc_score(y_test, y_pred_proba_pos)
                            
                            # Normalized AUC Score
                            normalized_auc = (auc - 0.5) / 0.5
                            
                            # Calculate ROC curve and Youden's index
                            fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba_pos)
                            
                            # Calculate Youden's index
                            youden_index = tpr - fpr
                            optimal_idx = np.argmax(youden_index)
                            optimal_threshold = thresholds[optimal_idx]
                            max_youden_index = youden_index[optimal_idx]
                            
                            # Calculate precision and recall at optimal threshold
                            y_pred_optimal = (y_pred_proba_pos >= optimal_threshold).astype(int)
                            optimal_precision = precision_score(y_test, y_pred_optimal, zero_division=0)
                            optimal_recall = recall_score(y_test, y_pred_optimal, zero_division=0)
                            
                            # Store ROC curve data
                            roc_data = {'fpr': fpr, 'tpr': tpr, 'thresholds': thresholds, 
                                      'optimal_idx': optimal_idx}
                        else:
                            auc = 0.5
                            normalized_auc = 0.0
                            optimal_threshold = 0.5
                            max_youden_index = 0.0
                            optimal_precision = 0.0
                            optimal_recall = 0.0
                            roc_data = None
                    except:
                        auc = 0.5
                        normalized_auc = 0.0
                        optimal_threshold = 0.5
                        max_youden_index = 0.0
                        optimal_precision = 0.0
                        optimal_recall = 0.0
                        roc_data = None
                    
                    # Store binary-specific metrics
                    results[model_name]['normalized_auc'].append(normalized_auc)
                    results[model_name]['optimal_threshold'].append(optimal_threshold)
                    results[model_name]['optimal_precision'].append(optimal_precision)
                    results[model_name]['optimal_recall'].append(optimal_recall)
                    results[model_name]['youden_index'].append(max_youden_index)
                    results[model_name]['roc_curves'].append(roc_data)
                    
                else:
                    # Multiclass classification metrics
                    try:
                        # Macro-averaged AUC (one-vs-rest)
                        auc = roc_auc_score(y_test, y_pred_proba, multi_class='ovr', average='macro')
                    except:
                        auc = 0.5
                
                # Standard metrics (work for both binary and multiclass)
                precision = precision_score(y_test, y_pred, zero_division=0, average='macro')
                recall = recall_score(y_test, y_pred, zero_division=0, average='macro')
                accuracy = accuracy_score(y_test, y_pred)
                
                # NEW: Additional multiclass metrics
                f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)
                macro_precision = precision_score(y_test, y_pred, average='macro', zero_division=0)
                macro_recall = recall_score(y_test, y_pred, average='macro', zero_division=0)
                weighted_precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
                weighted_recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
                
                # Per-class metrics
                per_class_report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
                
                # Confusion matrix
                conf_matrix = confusion_matrix(y_test, y_pred)
                
                # Store all results
                results[model_name]['auc'].append(auc)
                results[model_name]['precision'].append(precision)
                results[model_name]['recall'].append(recall)
                results[model_name]['accuracy'].append(accuracy)
                results[model_name]['f1_score'].append(f1)
                results[model_name]['macro_precision'].append(macro_precision)
                results[model_name]['macro_recall'].append(macro_recall)
                results[model_name]['weighted_precision'].append(weighted_precision)
                results[model_name]['weighted_recall'].append(weighted_recall)
                results[model_name]['per_class_metrics'].append(per_class_report)
                results[model_name]['confusion_matrices'].append(conf_matrix)
                results[model_name]['predictions'].append(y_pred)
                results[model_name]['y_true'].append(y_test)
                results[model_name]['pred_probabilities'].append(y_pred_proba)
                
                # Output metrics
                if is_binary:
                    print(f"   {model_name:12}: AUC={auc:.3f}, Norm_AUC={normalized_auc:.3f}, "
                          f"Prec={precision:.3f}, Rec={recall:.3f}, Acc={accuracy:.3f}")
                    print(f"   {'':12}  Opt_Thresh={optimal_threshold:.3f}, "
                          f"Opt_Prec={optimal_precision:.3f}, Opt_Rec={optimal_recall:.3f}, "
                          f"Youden={max_youden_index:.3f}")
                else:
                    print(f"   {model_name:12}: AUC={auc:.3f}, F1={f1:.3f}, "
                          f"Prec={precision:.3f}, Rec={recall:.3f}, Acc={accuracy:.3f}")
                    print(f"   {'':12}  W_Prec={weighted_precision:.3f}, "
                          f"W_Rec={weighted_recall:.3f}")
                
            except Exception as e:
                print(f"   ❌ {model_name} failed: {str(e)}")
        
        fold += 1
    
    # Return results with class mapping information
    return results, feature_cols, class_mapping, reverse_mapping

In [8]:
df_machine_98 = pd.read_csv("../../data/azure_pm/lag_features/machine_98_lag_features.csv")
df_machine_98.drop(columns=["datetime", "failure", "comp"], inplace=True)

validation_results, feature_names, class_mapping, reverse_mapping = walk_forward_validation(df_machine_98, n_splits=3)

🚀 Starting Walk-Forward Validation with 3 splits...
📊 Dataset info:
   - Total samples: 8,757
   - Features: 61
   - Original target classes: [0, 1, 2, 3, 4]
   - Remapped target classes: [0, 1, 2, 3, 4]
   - Class mapping: {0: 0, 1: 1, 2: 2, 3: 3, 4: 4}
   - Number of classes: 5
   - Target distribution: {0: 8409, 3: 122, 4: 78, 1: 75, 2: 73}
🎯 Classification type: MULTICLASS

📅 Fold 1/3
------------------------------
   Train: 2,190 samples, classes: [0, 1, 3, 4]
   Test:  2,189 samples, classes: [0, 1, 2, 3, 4]
   Train distribution: {0: 2116, 1: 25, 4: 25, 3: 24}
   Test distribution: {0: 2039, 4: 53, 3: 48, 1: 25, 2: 24}
   RandomForest: AUC=0.500, F1=0.363, Prec=0.461, Rec=0.325, Acc=0.943
                 W_Prec=0.907, W_Rec=0.943
   LightGBM    : AUC=0.500, F1=0.513, Prec=0.662, Rec=0.537, Acc=0.955
                 W_Prec=0.949, W_Rec=0.955
   ❌ XGBoost failed: Invalid classes inferred from unique values of `y`.  Expected: [0 1 2 3], got [0 1 3 4]
   CatBoost    : AUC=0.500, F

In [19]:
from sklearn.model_selection import TimeSeriesSplit

# Get predicted probabilities for CatBoost from the last fold
catboost_probs = validation_results['CatBoost']['pred_probabilities'][-1]
catboost_preds = validation_results['CatBoost']['predictions'][-1]
catboost_y_true = validation_results['CatBoost']['y_true'][-1]

# Uncertainty: smallest difference between top two predicted probabilities for each row
probs_sorted = np.sort(catboost_probs, axis=1)
uncertainty = probs_sorted[:, -1] - probs_sorted[:, -2]

# Get indices of top 10 most uncertain predictions (smallest difference)
top_uncertain_idx = np.argsort(uncertainty)[:10]

# Get the corresponding rows from the test set of the last fold
# Find the test indices used in the last fold

exclude_cols = ['datetime', 'machineID', 'failure', 'binary_failure', 'target']
feature_cols = [col for col in df_machine_98.columns if col not in exclude_cols]
X = df_machine_98[feature_cols].fillna(0)
y = df_machine_98['target']

tscv = TimeSeriesSplit(n_splits=3)
splits = list(tscv.split(X))
_, test_idx = splits[-1]

# Select the top 10 uncertain rows from the original dataframe
uncertain_rows = df_machine_98.iloc[np.array(test_idx)[top_uncertain_idx]]
uncertain_rows['predicted_class'] = catboost_preds[top_uncertain_idx]
uncertain_rows['true_class'] = catboost_y_true.iloc[top_uncertain_idx].values
uncertain_rows['uncertainty'] = uncertainty[top_uncertain_idx]

uncertain_rows.head(10)

,volt,rotate,pressure,vibration,errorID,target,volt_lag_1h,volt_lag_6h,volt_lag_12h,volt_lag_24h,...,maint_count_24h,hour,day_of_week,is_weekend,is_working_hours,hours_since_maint,hours_since_error,predicted_class,true_class,uncertainty
7380,0.505148,0.545656,0.363842,0.556224,4,0,0.582573,0.572114,0.484642,0.314689,...,0.0,23,1,0,0,185,0,1,0,0.000607
7402,0.682885,0.405996,0.427930,0.335144,0,0,0.696536,0.654245,0.408794,0.460949,...,0.0,21,2,0,0,207,22,0,0,0.001614
7116,0.561936,0.769525,0.319857,0.391553,0,0,0.794844,0.432070,0.605475,0.573846,...,0.0,0,5,1,0,282,15,1,0,0.003151
7381,0.554738,0.497432,0.440765,0.456308,0,0,0.505148,0.751000,0.567950,0.296440,...,0.0,0,2,0,0,186,1,1,0,0.006836
7117,0.599726,0.603804,0.376324,0.446307,0,0,0.561936,0.529094,0.638475,0.520843,...,0.0,1,5,1,0,283,16,0,0,0.019604
7388,0.441706,0.678264,0.419133,0.370211,0,0,0.788279,0.527877,0.572110,0.547401,...,0.0,7,2,0,0,193,8,0,0,0.026995
7384,0.503071,0.389919,0.401212,0.337249,0,0,0.723353,0.460949,0.890419,0.689231,...,0.0,3,2,0,0,189,4,0,0,0.028329
7120,0.696791,0.631021,0.480624,0.592071,0,0,0.487248,0.753932,0.471620,0.410252,...,0.0,4,5,1,0,286,19,1,0,0.035215
7403,0.708713,0.712999,0.573285,0.691103,0,0,0.682885,0.521130,0.807291,0.582573,...,0.0,22,2,0,0,208,23,0,0,0.035791
7119,0.487248,0.624287,0.491746,0.714915,0,0,0.376070,0.506921,0.489891,0.486000,...,0.0,3,5,1,0,285,18,0,0,0.039373


In [26]:
# Get feature names (excluding target and non-feature columns)
exclude_cols = ['datetime', 'machineID', 'failure', 'binary_failure', 'target']
feature_cols = [col for col in df_machine_98.columns if col not in exclude_cols]

# Get the CatBoost model from the last fold
catboost_model = validation_results['CatBoost']

# Get the last fold's test set indices
_, test_idx = splits[-1]
X_test = df_machine_98.iloc[test_idx][feature_cols].fillna(0)

# Use SHAP values for the last fold (already computed)
shap_vals = shap_values[-1]  # shape: (n_samples, n_features, n_classes)

# For multiclass, sum absolute SHAP values across all classes
# shap_importance = np.abs(shap_vals).sum(axis=2).mean(axis=0)
print(shap_vals.shape)  # Check the shape

if shap_vals.ndim == 3:
    # Multiclass: (n_samples, n_features, n_classes)
    shap_importance = np.abs(shap_vals).sum(axis=2).mean(axis=0)
elif shap_vals.ndim == 2:
    # Binary/classical: (n_samples, n_features)
    shap_importance = np.abs(shap_vals).mean(axis=0)
else:
    raise ValueError("Unexpected SHAP values shape: {}".format(shap_vals.shape))



print("feature_cols:", len(feature_cols))
print("shap_importance:", len(shap_importance))

# Ensure both arrays are the same length
min_len = min(len(feature_cols), len(shap_importance))
shap_importance_df = pd.DataFrame({
    'feature': feature_cols[:min_len],
    'mean_abs_shap': shap_importance[:min_len]
    # 'mean_abs_shap': shap_importance[:15]
}).sort_values('mean_abs_shap', ascending=False)

shap_importance_df.head()

# # Create a DataFrame for feature importance
# shap_importance_df = pd.DataFrame({
#     'feature': feature_cols,
#     'mean_abs_shap': shap_importance
# }).sort_values('mean_abs_shap', ascending=False)

# # Display the top 10 most important features
# shap_importance_df.h

(61, 5)
feature_cols: 61
shap_importance: 5


,feature,mean_abs_shap
3,vibration,0.087659
0,volt,0.082090
2,pressure,0.043982
1,rotate,0.040198
4,errorID,0.035920
